In [1]:
# Instalar pacotes necessários
!pip install tensorflow keras-tuner --quiet


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\pietr\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# Importações principais
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from keras_tuner.tuners import RandomSearch

import importlib.util
import sys

# Caminho completo para o arquivo
module_path = "../models/modelo_base_tuner.py"  # ajuste se necessário

# Nome arbitrário para o módulo
module_name = "modelo_base_tuner"

# Cria a especificação do módulo
spec = importlib.util.spec_from_file_location(module_name, module_path)
modelo_base_tuner = importlib.util.module_from_spec(spec)
sys.modules[module_name] = modelo_base_tuner
spec.loader.exec_module(modelo_base_tuner)

# Agora você pode usar:
build_model = modelo_base_tuner.build_model

In [3]:
# Gerar dados de exemplo (substitua pelo seu dataset real, se desejar)
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=1000, 
    n_features=20, 
    n_informative=15, 
    n_classes=2, 
    random_state=42
)


In [4]:
# Divisão em treino e teste e normalização
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
# Configuração do Keras Tuner para ajustar learning rate
input_dim = X_train.shape[1]

import os
import shutil
import tensorflow as tf

# Limpa o backend do TensorFlow
tf.keras.backend.clear_session()

# Cria um novo caminho isolado para o tuner
tuner_path = "/temp/"  # fora do projeto, evita conflitos

# Garante que o diretório seja removido se já existir
if os.path.exists(tuner_path):
    try:
        shutil.rmtree(tuner_path)
        print("Tuner antigo removido com sucesso.")
    except Exception as e:
        print("Erro ao apagar tuner antigo:", e)


from keras_tuner.tuners import RandomSearch

tuner = RandomSearch(
    lambda hp: build_model(hp, input_dim),
    objective="val_accuracy",
    max_trials=10,
    executions_per_trial=1,
    directory=tuner_path,
    project_name="modelo_base_tuner"
)


# Ajustar input_dim dinamicamente
for trial in tuner.oracle.get_space().space:
    if trial.name == "input_dim":
        trial.default = input_dim

# Executar busca
tuner.search(X_train, y_train, epochs=10, validation_split=0.2)

# Avaliação do melhor modelo
best_model = tuner.get_best_models(num_models=1)[0]
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print(f"Acurácia no teste: {test_acc:.4f}")

Trial 10 Complete [00h 00m 08s]
val_accuracy: 0.918749988079071

Best val_accuracy So Far: 0.9437500238418579
Total elapsed time: 00h 01m 04s


C:\Users\pietr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\saving\saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.9326 - loss: 0.2091
Acurácia no teste: 0.9300


In [6]:
# Apresenta os resultados
tuner.results_summary()


Results summary
Results in /temp/modelo_base_tuner
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 07 summary
Hyperparameters:
learning_rate: 0.004998741129453313
Score: 0.9437500238418579

Trial 06 summary
Hyperparameters:
learning_rate: 0.005180937737654028
Score: 0.9375

Trial 05 summary
Hyperparameters:
learning_rate: 0.0016069530933482345
Score: 0.9312499761581421

Trial 00 summary
Hyperparameters:
learning_rate: 0.0015462747018467995
Score: 0.925000011920929

Trial 01 summary
Hyperparameters:
learning_rate: 0.005518397014467231
Score: 0.925000011920929

Trial 09 summary
Hyperparameters:
learning_rate: 0.003128666297356665
Score: 0.918749988079071

Trial 02 summary
Hyperparameters:
learning_rate: 0.00012522252141090114
Score: 0.8500000238418579

Trial 04 summary
Hyperparameters:
learning_rate: 0.00036738608730941267
Score: 0.824999988079071

Trial 08 summary
Hyperparameters:
learning_rate: 0.00011519880606550622
Score: 0.78125

Trial 03 summary
Hyperp